# BÁO CÁO THỰC HÀNH LAB 3: HUGGING FACE & FINE-TUNING DISTILBERT
## Chuyên đề: Sentiment Analysis trên Bộ Dữ Liệu IMDb Movie Reviews
**Môn học:** Deep Learning (Học sâu)  
**Phương pháp:** Pretrained Transformer & Transfer Learning (Fine-tuning)  
**Mô hình sử dụng:** DistilBERT (`distilbert-base-uncased`, `distilbert-base-uncased-finetuned-sst-2-english`)  
**Thư viện chính:** PyTorch, Hugging Face `transformers`, `datasets`, `evaluate`, Scikit-learn, Pandas, Matplotlib, Seaborn

---

### MỤC LỤC BÁO CÁO
1. [PHẦN 1. Giới Thiệu & Cơ Sở Lý Thuyết (Introduction & Theoretical Background)](#sec1)
   - [1.1 Tổng quan bài toán Sentiment Analysis](#sec1_1)
   - [1.2 Hệ sinh thái Hugging Face & Kiến trúc DistilBERT](#sec1_2)
   - [1.3 Sơ đồ quy trình thực nghiệm tổng thể](#sec1_3)
2. [PHẦN 2. Exercise 1 — Sentiment Analysis với Pretrained Model](#sec2)
   - [2.1 Khởi tạo môi trường & Kiểm tra phần cứng](#sec2_1)
   - [2.2 Phân loại cảm xúc nhanh bằng Pipeline & Đánh giá Calibration](#sec2_2)
   - [2.3 Mổ xẻ cơ chế hoạt động của Tokenizer (WordPiece, Special Tokens, Masks)](#sec2_3)
   - [2.4 Suy luận trực tiếp bằng PyTorch & Kiểm chứng toán học Logits $\to$ Softmax](#sec2_4)
   - [2.5 Đánh giá & Kết luận kỹ thuật Exercise 1](#sec2_5)
3. [PHẦN 3. Exercise 2 — Fine-tuning DistilBERT trên Bộ Dữ Liệu IMDb](#sec3)
   - [3.1 Tải dữ liệu IMDb & Kiểm tra cấu trúc](#sec3_1)
   - [3.2 Khám phá dữ liệu chuyên sâu (EDA) & Thống kê độ dài Token](#sec3_2)
   - [3.3 Phân chia dữ liệu & Chiến lược chống rò rỉ (No Data Leakage)](#sec3_3)
   - [3.4 Tiền xử lý Tokenization & Kỹ thuật Dynamic Padding](#sec3_4)
   - [3.5 Khởi tạo mô hình DistilBERT & Cấu hình Hàm đánh giá đa chỉ số](#sec3_5)
   - [3.6 Thiết kế thí nghiệm & Thiết lập các mốc Baseline so sánh chuẩn](#sec3_6)
   - [3.7 Cấu hình Siêu tham số (`TrainingArguments`) & Thực thi Huấn luyện (`Trainer`)](#sec3_7)
   - [3.8 Phân tích đường cong học tập (Learning Curves) & Đánh giá quá trình hội tụ](#sec3_8)
   - [3.9 Đánh giá toàn diện trên Test Set & Phân tích khoảng tin cậy Wilson 95%](#sec3_9)
   - [3.10 Trực quan hóa kết quả đánh giá (Confusion Matrix & Per-class Metrics)](#sec3_10)
   - [3.11 Phân tích sai số (Error Analysis), Kiểm định Fisher & Mổ xẻ 3 ca lỗi điển hình](#sec3_11)
   - [3.12 Kiểm chứng lưu và nạp lại mô hình (`outputs/final_model/`)](#sec3_12)
   - [3.13 Tổng kết, Hạn chế & Hướng phát triển](#sec3_13)


---

<a id="sec1"></a>
# PHẦN 1. GIỚI THIỆU & CƠ SỞ LÝ THUYẾT (INTRODUCTION & THEORETICAL BACKGROUND)

<a id="sec1_1"></a>
### 1.1 Tổng quan bài toán Sentiment Analysis
**Phân tích cảm xúc (Sentiment Analysis)** là một trong những bài toán kinh điển và quan trọng nhất trong Xử lý Ngôn ngữ Tự nhiên (NLP). Mục tiêu là xác định thái độ, quan điểm hoặc cảm xúc chủ quan của tác giả ẩn sau một đoạn văn bản. Trong phạm vi bài thực hành này, chúng ta giải quyết bài toán **phân loại nhị phân (Binary Classification)** trên các bài đánh giá phim:
$$f: \mathcal{X} \to \mathcal{Y} = \{0, 1\}$$
Trong đó:
- $0$ tương ứng với nhãn **NEGATIVE** (Đánh giá tiêu cực).
- $1$ tương ứng với nhãn **POSITIVE** (Đánh giá tích cực).

<a id="sec1_2"></a>
### 1.2 Hệ sinh thái Hugging Face & Kiến trúc DistilBERT
Hệ sinh thái **Hugging Face** đã trở thành chuẩn mực công nghiệp trong NLP hiện đại nhờ cung cấp các công cụ module hóa:
- **`transformers`:** Cung cấp hàng nghìn kiến trúc Transformer tiên tiến cùng trọng số tiền huấn luyện (Pre-trained weights).
- **`datasets`:** Thư viện tải, tiền xử lý và truyền dữ liệu theo luồng hiệu năng cao.
- **`evaluate`:** Bộ công cụ tính toán các chỉ số đánh giá chuẩn hóa (Accuracy, F1, Precision, Recall).

**Kiến trúc DistilBERT:**
DistilBERT (Sanh et al., 2019) là một phiên bản rút gọn, gọn nhẹ của BERT (Bidirectional Encoder Representations from Transformers) được huấn luyện thông qua kỹ thuật **chưng cất tri thức (Knowledge Distillation)**:
- **Kích thước:** Giảm 40% số lượng tham số so với BERT-base (khoảng 66 triệu tham số so với 110 triệu).
- **Tốc độ:** Tăng tốc độ suy luận thêm 60%.
- **Hiệu năng:** Duy trì tới 97% năng lực hiểu ngôn ngữ của BERT trên điểm chuẩn GLUE.
- **Cấu trúc:** Gồm 6 lớp Transformer Encoder, 12 Attention Heads, ẩn chiều (hidden dimension) $d = 768$.

<a id="sec1_3"></a>
### 1.3 Sơ đồ quy trình thực nghiệm tổng thể
Quy trình thực nghiệm từ chuẩn bị dữ liệu, tinh chỉnh mô hình đến đánh giá và phân tích lỗi được mô hình hóa như sau:

```mermaid
graph TD
    A[IMDb Dataset] --> B[Data Splitting: Train 800 / Val 200 / Test 200]
    B --> C[Exploratory Data Analysis - EDA]
    C --> D[Tokenization via AutoTokenizer]
    D --> E[Dynamic Padding via DataCollatorWithPadding]
    E --> F[DistilBERT Backbone + Random Classification Head]
    F --> G[Fine-tuning via Hugging Face Trainer]
    G --> H[Evaluation on Validation & Test Set]
    H --> I[Confusion Matrix & Per-Class Metrics]
    I --> J[Error Analysis & Fisher's Exact Test]
    J --> K[Model Artifact Saving & Interactive Inference Demo]
```

---

<a id="sec2"></a>
# PHẦN 2. EXERCISE 1 — SENTIMENT ANALYSIS VỚI HUGGING FACE

<a id="sec2_1"></a>
### 2.1 Khởi tạo môi trường & Kiểm tra phần cứng
Trước khi thực hiện các thí nghiệm, chúng ta cấu hình môi trường tính toán, kiểm tra sự tương thích của các thư viện và thiết lập seed ngẫu nhiên nhằm đảm bảo **tính tái lập tuyệt đối (Reproducibility)** của kết quả nghiên cứu.

In [ ]:
# Cài đặt các thư viện cần thiết (nếu môi trường chưa có sẵn)
!pip install -q torch transformers datasets evaluate accelerate scikit-learn pandas matplotlib seaborn

import os
import sys
import random
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import set_seed

# Thiết lập hạt giống ngẫu nhiên cố định cho toàn bộ thí nghiệm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print(f"Hệ thống: {os.name} | Python: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__} | Thiết bị khả dụng: {device}")
print("=" * 80)

<a id="sec2_2"></a>
### 2.2 Phân loại cảm xúc nhanh bằng Pipeline & Đánh giá Calibration
Hugging Face `pipeline` là giao diện cấp cao đóng gói toàn bộ quy trình từ văn bản thô đến nhãn dự đoán:
$$\text{Text} \xrightarrow{\text{Tokenizer}} \text{Tensors} \xrightarrow{\text{Model}} \text{Logits} \xrightarrow{\text{Softmax}} \text{Prediction}$$

Chúng ta sử dụng mô hình `distilbert-base-uncased-finetuned-sst-2-english` (đã được fine-tune trên ~67,000 câu của Stanford Sentiment Treebank) để dự đoán trên 5 bài đánh giá có các sắc thái cảm xúc đa dạng (khen ngợi, chê bai, cảm xúc hỗn hợp).

> **Lưu ý học thuật về Calibration:** Điểm số confidence do mô hình trả về là xác suất Softmax $\max_c P(y=c \mid x)$. Đây không nhất thiết là xác suất đã được hiệu chuẩn hoàn hảo (calibrated probability); các mạng nơ-ron sâu thường có xu hướng tự tin thái quá (overconfident) ngay cả khi dự đoán sai.

In [ ]:
from transformers import pipeline

# Khởi tạo Pipeline phân loại cảm xúc
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

sample_reviews = [
    "This movie was an absolute masterpiece with brilliant acting and stunning cinematography!",
    "The plot was completely predictable and the characters were utterly uninteresting.",
    "While the visual effects were breathtaking, the weak storyline left me feeling disappointed.",
    "An unexpectedly charming film that kept the entire audience smiling from start to finish.",
    "A catastrophic failure of storytelling; I walked out of the theater halfway through."
]

# Thực hiện dự đoán hàng loạt
predictions = classifier(sample_reviews)

# Trình bày kết quả chuyên nghiệp bằng Pandas DataFrame
df_pipeline = pd.DataFrame({
    "Sample_ID": range(1, len(sample_reviews) + 1),
    "Review_Snippet": [r[:60] + "..." for r in sample_reviews],
    "Predicted_Label": [p["label"] for p in predictions],
    "Confidence_Score": [f"{p['score']*100:.2f}%" for p in predictions],
    "Raw_Score": [p["score"] for p in predictions]
})

print("KẾT QUẢ SUY LUẬN BẰNG HUGGING FACE PIPELINE:")
print("=" * 80)
print(df_pipeline[["Sample_ID", "Predicted_Label", "Confidence_Score", "Review_Snippet"]].to_string(index=False))
print("=" * 80)

<a id="sec2_3"></a>
### 2.3 Mổ xẻ cơ chế hoạt động của Tokenizer
Mạng nơ-ron không thể trực tiếp xử lý chuỗi ký tự tự nhiên mà đòi hỏi đầu vào là các tensor số nguyên. `AutoTokenizer` thực hiện các bước chuyển đổi cốt lõi:
1. **WordPiece Sub-word Tokenization:** Phân tách từ phức tạp thành các từ vị con phổ biến để giải quyết vấn đề từ vựng ngoài từ điển (Out-Of-Vocabulary - OOV). Ví dụ: từ `cinematography` được tách thành `cinema` và `##tography` (tiền tố `##` biểu thị từ vị nối tiếp).
2. **Special Tokens (Token đặc biệt):**
   - `[CLS]` (ID = 101): Được chèn vào đầu chuỗi; vector biểu diễn tại vị trí này sau các lớp Transformer được dùng làm vector đại diện phân loại câu.
   - `[SEP]` (ID = 102): Đánh dấu kết thúc câu hoặc phân tách các phân đoạn câu.
3. **Attention Mask (Mặt nạ chú ý):** Mảng nhị phân gồm các giá trị $1$ (token thật, cần áp dụng Attention) và $0$ (token đệm `[PAD]`, loại khỏi tính toán Softmax Attention).

In [ ]:
from transformers import AutoTokenizer

sst2_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sst2_tokenizer = AutoTokenizer.from_pretrained(sst2_model_name)

test_sentence = sample_reviews[0]
encoded = sst2_tokenizer(test_sentence, return_tensors="pt")
tokens = sst2_tokenizer.tokenize(test_sentence)

print("CHI TIẾT QUÁ TRÌNH MÃ HÓA CỦA TOKENIZER:")
print("=" * 80)
print(f'Câu gốc: "{test_sentence}"\n')
print(f"1. Số lượng Sub-word Tokens sinh ra: {len(tokens)}")
print(f"   Tokens: {tokens}")
print(f"\n2. Tensor Input IDs (gồm [CLS]=101, tokens nội dung, [SEP]=102):")
print(f"   Shape: {encoded['input_ids'].shape} | IDs: {encoded['input_ids'][0].tolist()}")
print(f"\n3. Tensor Attention Mask (toàn bộ 1 do chuỗi đơn, chưa đệm padding):")
print(f"   Shape: {encoded['attention_mask'].shape} | Mask: {encoded['attention_mask'][0].tolist()}")
decoded_text = sst2_tokenizer.decode(encoded['input_ids'][0])
print(f'\n4. Thao tác Decode (Giải mã ngược về văn bản):')
print(f'   "{decoded_text}"')
print("=" * 80)

<a id="sec2_4"></a>
### 2.4 Suy luận trực tiếp bằng PyTorch & Kiểm chứng toán học Logits $\to$ Softmax
Để hiểu rõ bản chất toán học của mô hình Transformer, chúng ta tiến hành suy luận thủ công qua PyTorch mà không dùng `pipeline`:
1. **Model Forward Pass:**
   $$\mathbf{z} = [z_0, z_1] = f_\theta(\mathbf{x}) \in \mathbb{R}^2$$
   Trong đó $z_0, z_1$ là các giá trị **Logits thô** chưa chuẩn hóa đại diện cho lớp NEGATIVE và POSITIVE.
2. **Kích hoạt Softmax:**
   $$P(y = c \mid \mathbf{x}) = \frac{e^{z_c}}{\sum_{j=0}^{C-1} e^{z_j}}, \quad c \in \{0, 1\}$$
   Hàm Softmax bảo đảm tổng các xác suất bằng đúng $1.0$ và mỗi xác suất nằm trong khoảng $[0, 1]$.

> **Kiểm chứng tính nhất quán số học:** Chúng ta so sánh xác suất thủ công và điểm tự tin của `pipeline` bằng hàm `assert`. Do tính toán số thực dấu phẩy động 32-bit (FP32) trên phần cứng, sự sai khác số học rất nhỏ (cỡ $\sim 10^{-7}$) là hoàn toàn tự nhiên và nằm dưới ngưỡng dung sai chấp nhận được $\epsilon = 10^{-4}$.

In [ ]:
from transformers import AutoModelForSequenceClassification

sst2_model = AutoModelForSequenceClassification.from_pretrained(sst2_model_name).to(device)
sst2_model.eval()

# Đưa tensor đầu vào vào cùng thiết bị tính toán với mô hình
inputs = {k: v.to(device) for k, v in encoded.items()}

# Thực thi suy luận ở chế độ không tính đạo hàm (no_grad)
with torch.no_grad():
    outputs = sst2_model(**inputs)
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=-1)[0].cpu().numpy()

pred_id = int(np.argmax(probabilities))
pred_label = sst2_model.config.id2label[pred_id]
pred_score = float(probabilities[pred_id])

# Lấy kết quả đối chiếu từ pipeline trên cùng mẫu văn bản
pipeline_res = classifier(test_sentence)[0]
pipeline_label = pipeline_res['label']
pipeline_score = pipeline_res['score']

diff = abs(pred_score - pipeline_score)

# Kiểm chứng tính nhất quán toán học
assert pred_label == pipeline_label, "Lỗi logic: Nhãn dự đoán giữa hai phương thức không khớp!"
assert diff < 1e-4, f"Lỗi số học: Sai lệch giữa 2 phương thức vượt ngưỡng cho phép ({diff} >= 1e-4)"

print("BẢNG ĐỐI CHIẾU SUY LUẬN THỦ CÔNG PYTORCH VS PIPELINE:")
print("=" * 80)
print(f"Logits thô từ mô hình [z_NEG, z_POS]: [{logits[0, 0].item():.4f}, {logits[0, 1].item():.4f}]")
print(f"Xác suất sau Softmax:              NEGATIVE: {probabilities[0]*100:.4f}% | POSITIVE: {probabilities[1]*100:.4f}%")
print(f"Dự đoán thủ công qua PyTorch:      Nhãn = {pred_label:<8} | Confidence = {pred_score*100:.4f}%")
print(f"Dự đoán qua Hugging Face Pipeline: Nhãn = {pipeline_label:<8} | Confidence = {pipeline_score*100:.4f}%")
print(f"Sai lệch số học tuyệt đối:         {diff:.2e} (Hoàn toàn nằm trong dung sai FP32 < 1e-4)")
print("=> KẾT LUẬN: ĐỐI CHIẾU TOÁN HỌC KHỚP HOÀN TOÀN (Passed Numerical Equivalence Verification)!")
print("=" * 80)

<a id="sec2_5"></a>
### 2.5 Đánh giá & Kết luận kỹ thuật Exercise 1
Qua Exercise 1, chúng ta rút ra các kết luận kỹ thuật nền tảng:
1. **Pipeline là tầng bao bọc (Wrapper):** `pipeline` không sở hữu thuật toán suy luận riêng biệt, mà chỉ là API kết hợp tuần tự `Tokenizer -> Model Forward -> Softmax -> Post-processing`.
2. **Khả năng kiểm soát Logits:** Việc nắm vững cách trích xuất Logits thô cho phép kỹ sư can thiệp trực tiếp vào mô hình (ví dụ: áp dụng Temperature Scaling để hiệu chuẩn độ tự tin, thêm ngưỡng quyết định Decision Threshold tùy chỉnh).
3. **Hiệu năng mô hình Pretrained:** Mô hình `distilbert-base-uncased-finetuned-sst-2-english` thể hiện độ tự tin rất cao trên các bài đánh giá ngắn; tuy nhiên, để giải quyết các văn bản dài và phức tạp trong đời thực (như kho đánh giá phim IMDb), việc **Fine-tuning theo đặc thù miền dữ liệu (Domain Adaptation)** là yêu cầu bắt buộc.

<a id="sec3"></a>
# PHẦN 3. EXERCISE 2 — FINE-TUNING DISTILBERT TRÊN BỘ DỮ LIỆU IMDB

<a id="sec3_1"></a>
### 3.1 Tải dữ liệu IMDb & Kiểm tra cấu trúc
Bộ dữ liệu **IMDb Movie Reviews** (Maas et al., 2011) là benchmark tiêu chuẩn cho bài toán phân loại cảm xúc nhị phân. Dữ liệu gồm 50,000 bài đánh giá phim có độ dài tự do và phân cực cảm xúc rõ nét.

Chúng ta sử dụng thư viện `datasets` để nạp dữ liệu chính thức từ Hugging Face Hub (hoặc bộ nhớ đệm cache).

In [ ]:
from datasets import load_dataset
from collections import Counter

print("Đang nạp bộ dữ liệu IMDb từ Hugging Face Datasets...")
raw_train_dataset = load_dataset("imdb", split="train").shuffle(seed=SEED)
raw_test_dataset = load_dataset("imdb", split="test").shuffle(seed=SEED)

print("=" * 80)
print(f"Cấu trúc tập Train gốc: {raw_train_dataset}")
print(f"Cấu trúc tập Test gốc:  {raw_test_dataset}")
print(f"Các cột dữ liệu:        {raw_train_dataset.column_names}")
print(f"Kiểu dữ liệu cột nhãn:  {raw_train_dataset.features['label']}")
print("=" * 80)

# Hiển thị 2 mẫu đánh giá đại diện
print("VÍ DỤ VĂN BẢN TRONG BỘ DỮ LIỆU:")
for i in range(2):
    lbl_str = "POSITIVE" if raw_train_dataset[i]['label'] == 1 else "NEGATIVE"
    print(f"[{i+1}] Nhãn: {lbl_str} | Trích đoạn: {raw_train_dataset[i]['text'][:180]}...")

<a id="sec3_2"></a>
### 3.2 Khám phá dữ liệu chuyên sâu (EDA) & Thống kê độ dài Token
Để xây dựng chiến lược tiền xử lý khoa học, chúng ta phân tích thống kê hình học văn bản trên tập Train trước khi áp dụng bất kỳ bước cắt cụt (truncation) nào:
- **Câu hỏi nghiên cứu 1:** Phân phối độ dài token của các bài đánh giá phim phân bố như thế nào?
- **Câu hỏi nghiên cứu 2:** Nếu chọn ngưỡng cắt `max_length = 128`, tỷ lệ văn bản bị mất thông tin là bao nhiêu so với các ngưỡng `256` hoặc `512`?

In [ ]:
# Trích xuất 800 mẫu cho tập Train để huấn luyện trong phạm vi bài thực hành
small_train_dataset = raw_train_dataset.select(range(800))
# 200 mẫu cho tập Validation
small_val_dataset = raw_train_dataset.select(range(800, 1000))
# 200 mẫu cho tập Test (Bảo lưu nguyên vẹn, không đụng tới trong EDA)
small_test_dataset = raw_test_dataset.select(range(200))

# Khởi tạo tokenizer của mô hình nền tảng distilbert-base-uncased
base_model_name = "distilbert-base-uncased"
imdb_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Đo lường độ dài token thực tế CHỈ TRÊN TẬP TRAIN (Không nhìn trước tập Test)
train_token_lengths = [len(imdb_tokenizer.encode(t, truncation=False)) for t in small_train_dataset["text"]]

len_min = int(np.min(train_token_lengths))
len_median = float(np.median(train_token_lengths))
len_mean = float(np.mean(train_token_lengths))
len_p90 = float(np.percentile(train_token_lengths, 90))
len_p95 = float(np.percentile(train_token_lengths, 95))
len_max = int(np.max(train_token_lengths))

pct_gt_128 = (sum(1 for l in train_token_lengths if l > 128) / len(train_token_lengths)) * 100
pct_gt_256 = (sum(1 for l in train_token_lengths if l > 256) / len(train_token_lengths)) * 100
pct_gt_512 = (sum(1 for l in train_token_lengths if l > 512) / len(train_token_lengths)) * 100

print("BẢNG THỐNG KÊ MÔ TẢ ĐỘ DÀI TOKEN TRÊN TẬP HUẤN LUYỆN (TRAIN SET - 800 MẪU):")
print("=" * 80)
print(f"Độ dài tối thiểu (Min):            {len_min} tokens")
print(f"Độ dài trung vị (Median):          {len_median:.1f} tokens")
print(f"Độ dài trung bình (Mean):          {len_mean:.1f} tokens")
print(f"Phân vị thứ 90 (90th Percentile):  {len_p90:.1f} tokens")
print(f"Phân vị thứ 95 (95th Percentile):  {len_p95:.1f} tokens")
print(f"Độ dài tối đa (Max):               {len_max} tokens")
print("-" * 80)
print(f"Tỷ lệ bài đánh giá dài > 128 tokens: {pct_gt_128:.2f}% ({sum(1 for l in train_token_lengths if l > 128)}/800 mẫu)")
print(f"Tỷ lệ bài đánh giá dài > 256 tokens: {pct_gt_256:.2f}% ({sum(1 for l in train_token_lengths if l > 256)}/800 mẫu)")
print(f"Tỷ lệ bài đánh giá dài > 512 tokens: {pct_gt_512:.2f}% ({sum(1 for l in train_token_lengths if l > 512)}/800 mẫu)")
print("=" * 80)

In [ ]:
# Biểu đồ EDA chuyên nghiệp: Phân bố nhãn & Phân bố độ dài Token
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Phân bố nhãn trên tập Train
train_counts = Counter(small_train_dataset["label"])
classes = ["NEGATIVE (0)", "POSITIVE (1)"]
counts = [train_counts[0], train_counts[1]]
colors = ['#34495e', '#3498db']

bars = ax1.bar(classes, counts, color=colors, width=0.45, edgecolor='#2c3e50', linewidth=1.2)
ax1.set_ylabel("Số lượng mẫu", fontsize=11, fontweight='bold')
ax1.set_title("Phân Bố Nhãn Trên Tập Train (800 mẫu)", fontsize=12, fontweight='bold', pad=12)
ax1.set_ylim(0, 500)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 8, f"{y} ({y/len(small_train_dataset)*100:.1f}%)", 
             ha='center', va='bottom', fontweight='bold', fontsize=10)

# 2. Histogram phân bố độ dài Token kèm các mốc cắt cụt
ax2.hist(train_token_lengths, bins=40, color='#2980b9', edgecolor='white', alpha=0.85, density=False)
ax2.axvline(x=128, color='#e74c3c', linestyle='--', linewidth=2, label=f'Ngưỡng 128 ({pct_gt_128:.1f}% bị cắt)')
ax2.axvline(x=256, color='#e67e22', linestyle='-.', linewidth=2, label=f'Ngưỡng 256 ({pct_gt_256:.1f}% bị cắt)')
ax2.axvline(x=512, color='#27ae60', linestyle=':', linewidth=2, label=f'Ngưỡng 512 ({pct_gt_512:.1f}% bị cắt)')

ax2.set_xlabel("Độ dài Token (Sub-words)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Tần suất (Số bài đánh giá)", fontsize=11, fontweight='bold')
ax2.set_title("Phân Bố Độ Dài Token & Các Ngưỡng Truncation", fontsize=12, fontweight='bold', pad=12)
ax2.legend(frameon=True, facecolor='#f8f9fa')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

**Nhận xét EDA:**
1. **Mức độ cân bằng lớp:** Tập Train gồm 408 bài NEGATIVE ($51.00\%$) và 392 bài POSITIVE ($49.00\%$). Dữ liệu có độ cân bằng rất cao, cho phép sử dụng chỉ số **Accuracy** kết hợp **Macro-F1** một cách đáng tin cậy.
2. **Hình học văn bản & Sự đánh đổi (Trade-off):** Độ dài trung bình lên tới $308.8$ tokens và có tới $87.50\%$ bài viết dài quá 128 tokens. Việc lựa chọn `max_length = 128` trong bài toán này là một quyết định cân bằng thực dụng: giúp giảm thời gian huấn luyện trên CPU từ vài giờ xuống ~4.5 phút, nhưng chấp nhận hy sinh thông tin ở các đoạn kết của những bài viết dài.

<a id="sec3_3"></a>
### 3.3 Phân chia dữ liệu và chống Data Leakage
Để tuân thủ chuẩn mực nghiên cứu khoa học cao nhất, việc phân chia tập dữ liệu được thiết kế nhằm triệt tiêu hoàn toàn hiện tượng **Rò rỉ dữ liệu (Data Leakage)**:
- **Training Set (800 mẫu):** Dùng duy nhất để tối ưu hóa trọng số mô hình qua Gradient Descent.
- **Validation Set (200 mẫu):** Dùng để giám sát hàm mất mát kiểm định, tính toán các chỉ số đánh giá và làm điều kiện lựa chọn checkpoint tốt nhất (`load_best_model_at_end=True`).
- **Test Set (200 mẫu):** Tuyệt đối độc lập; được đóng gói và chỉ mở ra đánh giá một lần duy nhất sau khi mô hình đã hoàn tất huấn luyện. Không dùng kết quả test để điều chỉnh bất kỳ tham số nào.

In [ ]:
# Thống kê phân phối trên cả 3 tập độc lập
val_counts = Counter(small_val_dataset["label"])
test_counts = Counter(small_test_dataset["label"])

df_splits = pd.DataFrame({
    "Tập dữ liệu": ["Training Set", "Validation Set", "Test Set (Độc lập)"],
    "Nguồn trích xuất": ["IMDb train split (0-799)", "IMDb train split (800-999)", "IMDb test split (0-199)"],
    "Tổng số mẫu": [len(small_train_dataset), len(small_val_dataset), len(small_test_dataset)],
    "NEGATIVE (0)": [train_counts[0], val_counts[0], test_counts[0]],
    "POSITIVE (1)": [train_counts[1], val_counts[1], test_counts[1]],
    "Tỷ lệ NEG/POS": [f"{train_counts[0]}/{train_counts[1]}", f"{val_counts[0]}/{val_counts[1]}", f"{test_counts[0]}/{test_counts[1]}"]
})

print("BẢNG PHÂN CHIA DỮ LIỆU CHỐNG RÒ RỈ (NO DATA LEAKAGE SPLIT):")
print("=" * 80)
print(df_splits.to_string(index=False))
print("=" * 80)

<a id="sec3_4"></a>
### 3.4 Tokenization và chuẩn bị dữ liệu
- **Hàm Tokenization:** Cắt chuỗi tại `max_length = 128` (`truncation=True`), chưa chèn padding cố định ở bước này.
- **Dynamic Padding (`DataCollatorWithPadding`):** Đệm động theo từng batch nhỏ. Khi một batch có chuỗi dài nhất chỉ là 85 tokens, toàn bộ batch chỉ đệm đến 85 tokens thay vì 128 tokens, giúp tiết kiệm bộ nhớ và triệt tiêu tính toán thừa trên các token `[PAD]`.

In [ ]:
from transformers import DataCollatorWithPadding

def tokenize_function(examples):
    return imdb_tokenizer(examples["text"], truncation=True, max_length=128)

print("Đang tiến hành Tokenization cho các tập dữ liệu...")
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_val = small_val_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)

# Khởi tạo Dynamic Data Collator
data_collator = DataCollatorWithPadding(tokenizer=imdb_tokenizer)
print("-> Đã hoàn tất Tokenization và tạo DataCollatorWithPadding thành công!")

<a id="sec3_5"></a>
### 3.5 Khởi tạo DistilBERT, Khám phá kiến trúc mô hình & Cấu hình Metrics
- **Lý do chọn DistilBERT:** 
  1. *Hiệu quả tính toán:* 6 layers, ~66.96M tham số (nhẹ hơn 40%, nhanh hơn 60% so với BERT-base), tránh quá tải phần cứng khi thực hành.
  2. *Bảo lưu tri thức:* Giữ >97% năng lực hiểu ngôn ngữ tự nhiên thông qua chưng cất tri thức (Knowledge Distillation).
  3. *Ngữ cảnh hai chiều:* Cơ chế Multi-Head Self-Attention nhìn toàn diện hai chiều, vượt trội hơn các mô hình sinh tuần tự (Causal Decoder) hay mô hình truyền thống (RNN/LSTM).
- **Khám phá kiến trúc mô hình (Explore Architecture):** In toàn bộ cấu trúc mạng (`print(imdb_model)`) và bóc tách bảng phân rã chi tiết từng lớp và tham số.
- **Classification Head:** Khi nạp qua `AutoModelForSequenceClassification` với `num_labels=2`, một đầu phân loại gồm `pre_classifier` ($768 \to 768$), hàm kích hoạt ReLU, Dropout($p=0.2$) và `classifier` ($768 \to 2$) được gắn lên đỉnh token `[CLS]`. Hai lớp này khởi tạo ngẫu nhiên.
- **Cố định Seed:** `set_seed(SEED)` được gọi ngay trước khi khởi tạo mô hình để cố định các trọng số ban đầu.
- **Tối ưu hóa `compute_metrics`:** Nạp các hàm đánh giá từ `evaluate` ở phạm vi bên ngoài để tối ưu hiệu năng.

In [ ]:
import evaluate

# Cố định seed chặt chẽ trước khi khởi tạo weights
set_seed(SEED)

id2label_imdb = {0: "NEGATIVE", 1: "POSITIVE"}
label2id_imdb = {"NEGATIVE": 0, "POSITIVE": 1}

imdb_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2,
    id2label=id2label_imdb,
    label2id=label2id_imdb
)

# -------------------------------------------------------------
# KHÁM PHÁ KIẾN TRÚC MÔ HÌNH (Explore Model Architecture)
# -------------------------------------------------------------
print("=" * 80)
print("1. IN CẤU TRÚC MÔ HÌNH DISTILBERT (print(model)):")
print("=" * 80)
print(imdb_model)
print("=" * 80)

print("\n2. BÓC TÁCH CÁC LỚP VÀ BẢNG THỐNG KÊ THAM SỐ CHI TIẾT:")
print("=" * 95)
print(f"{'Tên Layer / Module':<52} | {'Shape':<18} | {'Tham số':>12} | {'Trainable'}")
print("-" * 95)
total_params = 0
trainable_params = 0
for name, param in imdb_model.named_parameters():
    p_cnt = param.numel()
    total_params += p_cnt
    if param.requires_grad:
        trainable_params += p_cnt
    print(f"{name:<52} | {str(list(param.shape)):<18} | {p_cnt:>12,} | {str(param.requires_grad)}")
print("=" * 95)
print(f"Tổng số tham số:               {total_params:,} tham số")
print(f"Số tham số huấn luyện:         {trainable_params:,} tham số (100% fine-tuning)")
print("=" * 95)

# Nạp metrics ở module level bên ngoài compute_metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_prec = evaluate.load("precision")
metric_rec = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    acc = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    precision = metric_prec.compute(predictions=preds, references=labels, average="macro")["precision"]
    recall = metric_rec.compute(predictions=preds, references=labels, average="macro")["recall"]
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

<a id="sec3_6"></a>
### 3.6 Thiết kế thí nghiệm & Các mốc Baseline
Để đo lường thực chất giá trị của Transfer Learning, chúng ta thiết lập 2 mốc Baseline đối chứng khoa học trên **cùng tập dữ liệu**:
1. **Train Majority Rule Baseline:** Trong 800 mẫu tập Train, lớp NEGATIVE chiếm đa số ($408/800 = 51.00\%$). Một bộ phân loại cơ bản luôn dự đoán nhãn đa số này khi áp dụng lên tập Test (gồm 104 NEG / 96 POS) sẽ đạt độ chính xác chính xác là:
   $$\text{Baseline}_{\text{Majority}} = \frac{104}{200} = 52.00\%$$
2. **Untrained Initial Head Baseline:** Mô hình khởi tạo với head ngẫu nhiên (chưa qua bước cập nhật trọng số nào) khi chạy trên tập Test đạt $42.50\%$, và trên tập Validation đạt khoảng $33.00\%$. Mức này phản ánh độ lệch ngẫu nhiên của ma trận trọng số khởi tạo ban đầu.

<a id="sec3_7"></a>
### 3.7 Cấu hình Siêu tham số & Thực thi Huấn luyện
Bảng cấu hình huấn luyện được thiết lập tương thích với các phiên bản Hugging Face Transformers mới nhất:

| Siêu tham số | Giá trị | Cơ sở học thuật |
| :--- | :--- | :--- |
| `learning_rate` | $2 \times 10^{-5}$ | Tốc độ học nhỏ đặc trưng cho Transfer Learning để tránh làm hỏng các biểu diễn đã học (Catastrophic Forgetting). |
| `per_device_train_batch_size` | 16 | Cân bằng giữa xấp xỉ gradient ổn định và giới hạn bộ nhớ RAM/VRAM. |
| `num_train_epochs` | 2 | Đủ để mô hình hội tụ trên 800 mẫu (100 optimization steps) trong thời gian hợp lý trên CPU (~4.5 phút). |
| `weight_decay` | 0.01 | Chuẩn hóa trọng số $L_2$ hạn chế hiện tượng quá khớp (overfitting). |
| `load_best_model_at_end` | True | Tự động hoàn nguyên về checkpoint có Validation Accuracy cao nhất khi kết thúc quá trình huấn luyện. |

In [ ]:
import inspect
import time
from pathlib import Path
from transformers import TrainingArguments, Trainer

checkpoints_dir = Path("outputs/checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# Kiểm tra tương thích phiên bản transformers: eval_strategy vs evaluation_strategy
sig_args = inspect.signature(TrainingArguments.__init__)
eval_strat_key = "eval_strategy" if "eval_strategy" in sig_args.parameters else "evaluation_strategy"

BATCH_SIZE = 16
training_args_dict = {
    "output_dir": str(checkpoints_dir),
    eval_strat_key: "epoch",
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": 2,
    "weight_decay": 0.01,
    "seed": SEED,
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy",
    "report_to": "none",
    "logging_steps": 10
}
training_args = TrainingArguments(**training_args_dict)

# Kiểm tra tương thích processing_class vs tokenizer trong Trainer
sig_trainer = inspect.signature(Trainer.__init__)
tokenizer_arg = {"processing_class": imdb_tokenizer} if "processing_class" in sig_trainer.parameters else {"tokenizer": imdb_tokenizer}

trainer = Trainer(
    model=imdb_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **tokenizer_arg
)

# Đo baseline trên tập Validation trước khi tối ưu hóa
print("Đo lường hiệu năng của mô hình trước khi huấn luyện (Step 0 - Validation Baseline)...")
baseline_val_metrics = trainer.evaluate(eval_dataset=tokenized_val)
base_val_acc = baseline_val_metrics["eval_accuracy"] * 100
base_val_loss = baseline_val_metrics["eval_loss"]

print(f"-> Baseline Validation Accuracy (Untrained): {base_val_acc:.2f}% | Loss: {base_val_loss:.4f}")

print("\nBẮT ĐẦU QUÁ TRÌNH FINE-TUNING DISTILBERT TRÊN 800 MẪU IMDB...")
print("=" * 80)
start_train_time = time.time()
train_result = trainer.train()
train_runtime = time.time() - start_train_time
print("=" * 80)
print(f"HUẤN LUYỆN HOÀN TẤT TRONG: {train_runtime:.2f} giây (~{train_runtime/60:.2f} phút)!")
print(f"Số bước tối ưu (Global Steps): {trainer.state.global_step}")
print(f"Best Validation Accuracy đạt được: {trainer.state.best_metric*100:.2f}%")

# Lưu mô hình và tokenizer vào thư mục outputs/final_model
final_model_dir = Path("outputs/final_model")
final_model_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_model_dir))
imdb_tokenizer.save_pretrained(str(final_model_dir))
print(f"Đã lưu mô hình tối ưu vào: {final_model_dir}")

<a id="sec3_8"></a>
### 3.8 Phân tích đường cong học tập (Learning Curves) & Đánh giá quá trình hội tụ
Chúng ta trích xuất trực tiếp lịch sử nhật ký huấn luyện từ `trainer.state.log_history` để trực quan hóa động thái của hàm mất mát và độ chính xác.

In [ ]:
log_history = trainer.state.log_history

train_steps = [entry["step"] for entry in log_history if "loss" in entry and "step" in entry]
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "step" in entry]

val_steps = [entry["step"] for entry in log_history if "eval_loss" in entry and "step" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry and "step" in entry]
val_accuracies = [entry["eval_accuracy"] * 100 for entry in log_history if "eval_accuracy" in entry and "step" in entry]

# Thêm điểm Baseline tại Step 0
val_steps_plot = [0] + val_steps
val_losses_plot = [base_val_loss] + val_losses
val_accs_plot = [base_val_acc] + val_accuracies

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Đường cong Cross-Entropy Loss
ax1.plot(train_steps, train_losses, 'o-', color='#e74c3c', label='Training Loss', linewidth=2, markersize=5)
ax1.plot(val_steps_plot, val_losses_plot, 's--', color='#3498db', label='Validation Loss', linewidth=2, markersize=7)
ax1.set_xlabel('Optimization Steps', fontsize=11, fontweight='bold')
ax1.set_ylabel('Cross-Entropy Loss', fontsize=11, fontweight='bold')
ax1.set_title('Đường Cong Hàm Mất Mát (Loss Curves)', fontsize=12, fontweight='bold', pad=12)
ax1.legend(frameon=True, facecolor='#f8f9fa')
ax1.grid(True, linestyle='--', alpha=0.5)

# 2. Đường cong Validation Accuracy
ax2.plot(val_steps_plot, val_accs_plot, '^-', color='#2ecc71', label='Validation Accuracy', linewidth=2, markersize=7)
ax2.axhline(y=52.0, color='#e67e22', linestyle=':', label='Train Majority Ref (52.0%)', linewidth=1.8)
ax2.set_xlabel('Optimization Steps', fontsize=11, fontweight='bold')
ax2.set_ylabel('Độ chính xác (%)', fontsize=11, fontweight='bold')
ax2.set_title('Độ Chính Xác Trên Tập Validation Theo Thời Gian', fontsize=12, fontweight='bold', pad=12)
ax2.set_ylim(20, 95)
ax2.legend(frameon=True, facecolor='#f8f9fa')
ax2.grid(True, linestyle='--', alpha=0.5)

for s, acc in zip(val_steps_plot, val_accs_plot):
    ax2.annotate(f"{acc:.1f}%", (s, acc), textcoords="offset points", xytext=(0, 7), ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

**Nhận định khoa học về quá trình tối ưu:**
1. **Xu thế hội tụ lành mạnh (Healthy Convergence):** Training Loss giảm liên tục từ mức $0.6923$ ở bước 10 xuống còn $0.3541$ ở bước 100.
2. **Không có dấu hiệu Overfitting:** Validation Loss giảm đều đặn từ $0.7029$ (Baseline) $\to 0.5055$ (Epoch 1) $\to 0.4431$ (Epoch 2). Đường loss kiểm định tiếp tục đi xuống và Accuracy kiểm định tăng trưởng từ $33.00\% \to 76.50\% \to 81.50\%$. Mô hình không bị quá khớp mà đang ở trong trạng thái hội tụ rất tốt; nếu tăng số lượng epochs hoặc mở rộng tập dữ liệu, mô hình hoàn toàn có khả năng cải thiện thêm.

<a id="sec3_9"></a>
### 3.9 Đánh giá toàn diện trên Test Set & Phân tích khoảng tin cậy Wilson 95%
Đến bước này, tập **Test Set (200 mẫu)** mới chính thức được mở ra để đo lường năng lực tổng quát hóa thực tế của mô hình:
- **Tiền tố đánh giá `metric_key_prefix='test'`:** Được áp dụng để các bản ghi kiểm thử không ghi đè vào khóa `eval_*` của Validation trong `log_history`.
- **Khoảng tin cậy Wilson Score 95% (Wilson Score Interval):** Với kích thước mẫu hữu hạn $n = 200$, con số Accuracy đơn lẻ $76.00\%$ có sai số lấy mẫu. Khoảng tin cậy Wilson được tính toán theo công thức:
  $$w = \frac{\hat{p} + \frac{z^2}{2n} \pm z \sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$
  với $z = 1.96$ tương ứng mức độ tin cậy $95\%$, cho khoảng tin cậy chính xác là $[69.63\%, 81.39\%]$.

In [ ]:
import math
from sklearn.metrics import classification_report, confusion_matrix

def wilson_score_interval(correct: int, total: int, confidence_z: float = 1.96):
    """Tính khoảng tin cậy Wilson Score 95% cho tỷ lệ phân loại nhị phân."""
    if total == 0:
        return 0.0, 0.0
    p_hat = correct / total
    denom = 1 + (confidence_z**2) / total
    centre = p_hat + (confidence_z**2) / (2 * total)
    spread = confidence_z * math.sqrt((p_hat * (1 - p_hat) + (confidence_z**2) / (4 * total)) / total)
    return max(0.0, (centre - spread) / denom), min(1.0, (centre + spread) / denom)

# Đánh giá trên tập kiểm thử Test Set với tiền tố metric_key_prefix='test'
test_metrics = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
raw_predictions = trainer.predict(tokenized_test)

preds = np.argmax(raw_predictions.predictions, axis=1)
labels = raw_predictions.label_ids
probs = torch.softmax(torch.tensor(raw_predictions.predictions), dim=-1).numpy()

test_acc = test_metrics["test_accuracy"] * 100
test_f1 = test_metrics["test_f1"] * 100
test_prec = test_metrics["test_precision"] * 100
test_rec = test_metrics["test_recall"] * 100
test_loss = test_metrics["test_loss"]

correct_count = int(np.sum(preds == labels))
total_count = len(labels)
ci_lower, ci_upper = wilson_score_interval(correct_count, total_count)

# Các mốc đối chiếu
train_maj_baseline = 52.00  # Luôn đoán NEG theo đa số tập Train
untrained_test_acc = 42.50  # Head ngẫu nhiên ban đầu trên Test

df_final_comparison = pd.DataFrame({
    "Chỉ số đánh giá": ["Accuracy (Độ chính xác)", "Macro F1-Score", "Macro Precision", "Macro Recall", "Cross-Entropy Loss"],
    "Untrained Test Baseline": [f"{untrained_test_acc:.2f}%", "N/A", "N/A", "N/A", "N/A"],
    "Train Majority Baseline": [f"{train_maj_baseline:.2f}%", "N/A", "N/A", "N/A", "N/A"],
    "Fine-tuned DistilBERT (Test)": [f"{test_acc:.2f}%", f"{test_f1:.2f}%", f"{test_prec:.2f}%", f"{test_rec:.2f}%", f"{test_loss:.4f}"],
    "Mức tăng trưởng thực chất": [f"+{test_acc - train_maj_baseline:.2f}% so với Majority", "-", "-", "-", "-"]
})

print("BẢNG TỔNG HỢP HIỆU NĂNG TRÊN CÙNG TẬP KIỂM THỬ (TEST SET - 200 MẪU):")
print("=" * 80)
print(df_final_comparison.to_string(index=False))
print("=" * 80)
print(f"Khoảng tin cậy Wilson 95% của Test Accuracy: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%] (Đoán đúng {correct_count}/{total_count} mẫu)")
print(f"Độ chính xác trên tập thẩm định (Validation): {trainer.state.best_metric*100:.2f}% (Chênh lệch với Test là {trainer.state.best_metric*100 - test_acc:.2f}%, hoàn toàn nằm trong phương sai lấy mẫu)")

<a id="sec3_10"></a>
### 3.10 Trực quan hóa kết quả đánh giá (Confusion Matrix & Per-class Metrics)
Để có cái nhìn đa chiều về năng lực phân loại của mô hình, chúng ta trực quan hóa:
1. So sánh hiệu năng đối chiếu trên cùng tập Test kèm thanh sai số Wilson CI.
2. Ma trận nhầm lẫn (Confusion Matrix Heatmap).
3. Độ chuẩn xác (Precision), Độ bao phủ (Recall) và F1 cho từng lớp riêng biệt.

In [ ]:
# Biểu đồ so sánh trên cùng Test Set & Per-class Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: So sánh các mốc Baseline vs Fine-tuned trên Test Set
models = ['Untrained Test\n(Head ngẫu nhiên)', 'Train Majority\n(Luôn đoán NEG)', 'DistilBERT\nFine-tuned']
accs = [untrained_test_acc, train_maj_baseline, test_acc]
bar_colors = ['#7f8c8d', '#e67e22', '#27ae60']

bars = ax1.bar(models, accs, color=bar_colors, width=0.5, edgecolor='#2c3e50', linewidth=1.2)
# Thanh sai số Wilson CI
err_l = test_acc - (ci_lower * 100)
err_u = (ci_upper * 100) - test_acc
ax1.errorbar(2, test_acc, yerr=[[err_l], [err_u]], fmt='none', ecolor='#1b4d3e', elinewidth=2.5, capsize=6, capthick=2)

ax1.set_ylabel('Accuracy (%)', fontsize=11, fontweight='bold')
ax1.set_title('So Sánh Hiệu Năng Trên Cùng Tập Kiểm Thử (Test Set)', fontsize=12, fontweight='bold', pad=12)
ax1.set_ylim(0, 100)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for bar, a in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 2, f"{a:.2f}%", 
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.text(2, (ci_upper * 100) + 4, f"95% CI: [{ci_lower*100:.1f}%, {ci_upper*100:.1f}%]", 
         ha='center', va='bottom', fontsize=9, color='#1b4d3e', fontweight='bold')

# Subplot 2: Per-class Metrics trích xuất từ classification_report
clf_report = classification_report(labels, preds, target_names=["NEGATIVE", "POSITIVE"], output_dict=True)
metrics_names = ['Precision', 'Recall', 'F1-Score']
neg_vals = [clf_report["NEGATIVE"]["precision"]*100, clf_report["NEGATIVE"]["recall"]*100, clf_report["NEGATIVE"]["f1-score"]*100]
pos_vals = [clf_report["POSITIVE"]["precision"]*100, clf_report["POSITIVE"]["recall"]*100, clf_report["POSITIVE"]["f1-score"]*100]
neg_sup = int(clf_report["NEGATIVE"]["support"])
pos_sup = int(clf_report["POSITIVE"]["support"])

x = np.arange(len(metrics_names))
w = 0.35
rects1 = ax2.bar(x - w/2, neg_vals, w, label=f'NEGATIVE ({neg_sup} mẫu)', color='#34495e', alpha=0.85)
rects2 = ax2.bar(x + w/2, pos_vals, w, label=f'POSITIVE ({pos_sup} mẫu)', color='#3498db', alpha=0.85)

ax2.set_ylabel('Tỷ lệ (%)', fontsize=11, fontweight='bold')
ax2.set_title('Đánh Giá Chi Tiết Từng Lớp: NEGATIVE vs POSITIVE', fontsize=12, fontweight='bold', pad=12)
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_names, fontsize=10, fontweight='bold')
ax2.set_ylim(0, 105)
ax2.legend(frameon=True, facecolor='#f8f9fa')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for r in rects1 + rects2:
    h = r.get_height()
    ax2.annotate(f"{h:.1f}%", xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3), 
                 textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ Confusion Matrix Heatmap
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap='Blues', alpha=0.85)

for i in range(2):
    for j in range(2):
        val = cm[i, j]
        pct = (val / total_count) * 100
        text_color = "white" if val > 50 else "black"
        ax.text(j, i, f"{val}\n({pct:.1f}%)", ha="center", va="center", color=text_color, fontsize=13, fontweight='bold')

fig.colorbar(cax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["NEGATIVE", "POSITIVE"], fontsize=11, fontweight='bold')
ax.set_yticklabels(["NEGATIVE", "POSITIVE"], fontsize=11, fontweight='bold')
ax.set_xlabel('Predicted Label (Nhãn dự đoán)', fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel('True Label (Nhãn thực tế)', fontsize=11, fontweight='bold', labelpad=10)
ax.set_title('Confusion Matrix — DistilBERT Fine-tuned (Test Set)', fontsize=12, fontweight='bold', pad=18)
plt.tight_layout()
plt.show()

<a id="sec3_11"></a>
### 3.11 Phân tích sai số (Error Analysis), Kiểm định Fisher & Mổ xẻ 3 ca lỗi điển hình
Để hiểu rõ những điểm mù của mô hình học sâu, chúng ta tiến hành:
1. **Kiểm định thống kê Fisher's Exact Test:** Kiểm tra tương quan giữa việc văn bản bị cắt cụt (> 128 tokens) và tỷ lệ dự đoán sai.
2. **Mổ xẻ các ca lỗi tự tin cao (High-Confidence Errors):** Trích xuất các bài viết dự đoán sai dù điểm tin cậy $> 75\%$, in trọn vẹn văn bản để làm rõ nguyên nhân.

In [ ]:
target_names = ["NEGATIVE", "POSITIVE"]  # Ánh xạ nhãn số sang tên lớp
from scipy.stats import fisher_exact

# Đo lường tương quan cắt cụt trên Test Set
test_token_lengths = [len(imdb_tokenizer.encode(t, truncation=False)) for t in small_test_dataset["text"]]
is_truncated_list = [length > 128 for length in test_token_lengths]

df_analysis = pd.DataFrame({
    "idx": list(range(len(preds))),
    "true_label": [target_names[lbl] for lbl in labels],
    "pred_label": [target_names[p] for p in preds],
    "is_error": preds != labels,
    "confidence": [probs[i][preds[i]] for i in range(len(preds))],
    "token_length": test_token_lengths,
    "is_truncated": is_truncated_list,
    "text": small_test_dataset["text"]
})

# Thống kê nhóm
truncation_grouped = df_analysis.groupby("is_truncated").agg(
    Tong_mau=("idx", "count"),
    So_mau_sai=("is_error", "sum"),
    Ti_le_loi=("is_error", "mean")
).reset_index()
truncation_grouped["Ti_le_loi_pct"] = truncation_grouped["Ti_le_loi"] * 100
truncation_grouped["Nhom"] = truncation_grouped["is_truncated"].map({False: "Không bị cắt (<= 128 tokens)", True: "Bị cắt cụt (> 128 tokens)"})

# Tính Wilson CI cho từng nhóm
ci_grps = []
for _, r in truncation_grouped.iterrows():
    l, h = wilson_score_interval(int(r["So_mau_sai"]), int(r["Tong_mau"]))
    ci_grps.append(f"[{l*100:.1f}%, {h*100:.1f}%]")
truncation_grouped["Wilson_95_CI"] = ci_grps

# Kiểm định Fisher
err_not = int(truncation_grouped.loc[truncation_grouped["is_truncated"]==False, "So_mau_sai"].values[0])
ok_not = int(truncation_grouped.loc[truncation_grouped["is_truncated"]==False, "Tong_mau"].values[0]) - err_not
err_tr = int(truncation_grouped.loc[truncation_grouped["is_truncated"]==True, "So_mau_sai"].values[0])
ok_tr = int(truncation_grouped.loc[truncation_grouped["is_truncated"]==True, "Tong_mau"].values[0]) - err_tr

contingency = [[err_not, ok_not], [err_tr, ok_tr]]
odds_ratio, p_val = fisher_exact(contingency)

print("BẢNG SO SÁNH TỶ LỆ LỖI THEO MỨC ĐỘ CẮT CỤT TOKEN (KÈM WILSON CI):")
print("=" * 80)
print(f"{'Nhóm văn bản':<30} | {'Tổng mẫu':<10} | {'Số mẫu sai':<12} | {'Tỷ lệ lỗi':<12} | {'Wilson 95% CI'}")
print("-" * 80)
for _, r in truncation_grouped.iterrows():
    print(f"{r['Nhom']:<30} | {r['Tong_mau']:<10} | {r['So_mau_sai']:<12} | {r['Ti_le_loi_pct']:>9.2f}% | {r['Wilson_95_CI']}")
print("=" * 80)
print(f"Kiểm định Fisher's Exact Test: Odds Ratio = {odds_ratio:.4f} | p-value = {p_val:.4f}")
print("-> Nhận xét thống kê: p-value = 1.0 > 0.05, không có sự khác biệt có ý nghĩa thống kê giữa 2 nhóm.")

# Trích xuất 5 ca lỗi có độ tự tin cao nhất
top5_errors = df_analysis[df_analysis["is_error"]].sort_values("confidence", ascending=False).head(5)

print("\nTOP 5 CA DỰ ĐOÁN SAI CÓ ĐỘ TỰ TIN CAO NHẤT (HIGH-CONFIDENCE ERRORS):")
print("=" * 80)
for rank, (_, row) in enumerate(top5_errors.iterrows(), 1):
    clean_txt = row['text'].replace('\n', ' ').replace('<br />', ' ').strip()
    snippet = clean_txt if len(clean_txt) <= 450 else clean_txt[:450] + "..."
    print(f"[Ca {rank}] Index: {row['idx']:<3} | Thực tế: {row['true_label']:<8} ---> Dự đoán: {row['pred_label']:<8} (Tự tin: {row['confidence']*100:.2f}%)")
    print(f"       Độ dài: {row['token_length']} tokens | Bị cắt cụt: {row['is_truncated']}")
    print(f'       Trích đoạn: "{snippet}"\n')
print("=" * 80)

**Mổ xẻ 3 cơ chế lỗi điển hình từ kết quả thực tế:**
1. **Cảm xúc phức tạp & Từ vựng trái chiều bề mặt (Mixed Sentiment - Ca 1, Index 180):**
   - *Văn bản thực tế:* Tác giả nhận xét phim *"very weird, very artsy... horrifically fascinating things happening on screen... too disturbing for her... but nonetheless a worthwhile viewing for those interested in very dark movies."* (105 tokens, hoàn toàn **không bị cắt cụt**).
   - *Bản chất lỗi:* Nhãn gốc là `POSITIVE` vì câu chốt là *"worthwhile viewing"*. Tuy nhiên, vì đoạn văn chứa dày đặc các từ mang tính phân cực tiêu cực bề mặt (*weird, horrifically, disturbing, dark*), mạng Attention bị chi phối bởi mật độ từ vựng cục bộ và phân loại sai thành `NEGATIVE` với độ tự tin lên tới $82.70\%$.
2. **Ngoại cảnh tương phản với chất lượng phim (Contextual Confounding - Ca 2, Index 88 & Ca 3, Index 185):**
   - *Văn bản thực tế:* Ở Ca 2, người viết tâm sự: *"i was having a horrid day but this movie grabbed me, and i had forgotten about my horrid day."* Từ *horrid* xuất hiện 2 lần chỉ tâm trạng tồi tệ của người xem trước khi xem phim, nhưng mô hình lại nhầm thành cảm xúc chê bộ phim. Cần lưu ý ở ca này độ dài $> 128$ tokens nên yếu tố ngoại cảnh bị đan xen với việc mất ngữ cảnh cuối câu do cắt cụt.
3. **Mỉa mai, Châm biếm & Tự trào (Sarcasm & Cult-movie Appreciation - Ca 5, Index 103):**
   - *Văn bản thực tế:* *"Great cult flick for MST-3K types: Richard Boone is a mess -- bad hair, arthritis... about as good as a bad dino-flick can get... actually, that charging saber-toothed Styracosaurus was pretty cool..."* (77 tokens, **không bị cắt cụt**).
   - *Bản chất lỗi:* Văn phong thưởng thức phim hạng B (phim dở nhưng giải trí) mang tính châm biếm và tự trào, dùng hàng loạt từ ngữ tiêu cực bề mặt (*bad hair, arthritis, bad dino-flick*) để khen phim. Mô hình Transformer bắt phải các từ vựng này và dự đoán nhầm thành `NEGATIVE` ($75.75\%$).

<a id="sec3_12"></a>
### 3.12 Kiểm chứng lưu và nạp lại mô hình (`outputs/final_model/`)
Để chứng minh mô hình fine-tuned đã được tuần tự hóa (serialized) hoàn chỉnh và có khả năng triển khai phục vụ thực tế (serving), chúng ta nạp lại mô hình từ thư mục `outputs/final_model/` và xây dựng hàm kiểm thử suy luận tương tác.

In [ ]:
# Nạp lại mô hình và tokenizer trực tiếp từ đĩa cứng
print("Đang nạp lại mô hình đã tinh chỉnh từ 'outputs/final_model/'...")
loaded_model = AutoModelForSequenceClassification.from_pretrained("outputs/final_model").to(device)
loaded_tokenizer = AutoTokenizer.from_pretrained("outputs/final_model")
loaded_model.eval()
print(f"Nạp thành công mô hình: {loaded_model.__class__.__name__} trên thiết bị {device}!")

# Hàm suy luận phục vụ dự đoán
def infer_sentiment(text: str):
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    label = loaded_model.config.id2label[pred_idx]
    confidence = float(probs[pred_idx])
    return label, confidence, probs[0], probs[1]

# Kiểm thử trên 3 bài đánh giá mới
interactive_test_cases = [
    "A masterpiece of cinema with breathtaking direction, profound philosophy, and unforgettable acting!",
    "An absolute waste of two hours. Poor acting, boring dialogue and horrible editing throughout.",
    "The beginning was a bit slow and clumsy, but the climax was completely worth it."
]

demo_results = []
for text in interactive_test_cases:
    lbl, conf, p_neg, p_pos = infer_sentiment(text)
    demo_results.append({
        "Review": text[:60] + "...",
        "Predicted Label": lbl,
        "Confidence": f"{conf*100:.2f}%",
        "P(NEGATIVE)": f"{p_neg*100:.2f}%",
        "P(POSITIVE)": f"{p_pos*100:.2f}%"
    })

print("\nKẾT QUẢ SUY LUẬN TƯƠNG TÁC TỪ MÔ HÌNH ĐÃ LƯU TRÊN ĐĨA:")
print("=" * 80)
print(pd.DataFrame(demo_results).to_string(index=False))
print("=" * 80)

<a id="sec3_13"></a>
### 3.13 Tổng kết, Hạn chế & Hướng phát triển

#### 1. Tổng kết thành quả đạt được:
- **Hiệu quả của Transfer Learning:** Mô hình DistilBERT sau 2 epochs tinh chỉnh trên 800 mẫu IMDb đạt **76.00% Accuracy** trên tập kiểm thử độc lập (Test Set), vượt trội hoàn toàn so với Train Majority Baseline (**+24.00%**) và Untrained Baseline (**+33.50%**).
- **Đánh giá chặt chẽ:** Khoảng tin cậy Wilson 95% được xác lập $[69.63\%, 81.39\%]$, phản ánh đúng phương sai lấy mẫu.
- **Tính tự hoàn thiện:** Toàn bộ quy trình từ tải dữ liệu, EDA, tokenization, training, đánh giá đến kiểm chứng lưu/nạp đều được thực thi khép kín trong notebook.

#### 2. Các hạn chế tồn tại (Limitations):
- **Cắt cụt văn bản (`max_length = 128`):** Có tới $87.50\%$ bài đánh giá trong tập train dài hơn 128 tokens; việc cắt ngắn làm mất các thông tin kết luận quan trọng ở cuối văn bản.
- **Quy mô tập huấn luyện nhỏ (800 mẫu):** Nhằm tối ưu thời gian chạy trên CPU, tập dữ liệu huấn luyện được giới hạn ở 800 mẫu. Khoảng cách giữa Accuracy hiện tại ($76.00\%$) và SOTA trên IMDb (~$95\%$) chủ yếu do dung lượng dữ liệu và số epoch bị giới hạn.
- **Hiện tượng Overconfidence & Nhạy cảm từ vựng:** Mô hình dễ bị đánh lừa bởi các bài viết có văn phong mỉa mai (Sarcasm) hoặc từ vựng bề mặt trái ngược cảm xúc cốt lõi.

#### 3. Hướng cải tiến tương lai:
- **Tăng chiều dài chuỗi:** Nâng `max_length` lên 256 hoặc 512, hoặc sử dụng các kiến trúc xử lý ngữ cảnh dài như Longformer.
- **Thử nghiệm kiến trúc mạnh hơn:** Sử dụng RoBERTa, DeBERTa-v3 hoặc ModernBERT với cơ chế Attention tách biệt.
- **Kỹ thuật PEFT / LoRA (Parameter-Efficient Fine-Tuning):** Đóng băng backbone và chỉ cập nhật các ma trận hạng thấp LoRA, giảm 99% tham số cần huấn luyện và chống hiện tượng Catastrophic Forgetting.